# Ridge RT Analysis From Hidden States

This notebook rewrites the logic from `modeling/rt_ridge.py` into a step-by-step notebook workflow.

- 4 pre-response windows
- baseline / hidden / baseline plus hidden / shuffled-hidden control
- 5-fold outer CV with per-fold alpha selection
- inline result tables and plots only


In [3]:
from pathlib import Path

#import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

#plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


In [6]:
project_dir = Path.cwd().resolve().parent if Path.cwd().name == 'script_regression' else Path('/Users/hyijie/Desktop/biiigProject/stage2_YuYNet')
dataset_dir = project_dir / 'script_regression' 
latent_path = dataset_dir / 'temp_data' / 'latents_full.npz'
metadata_path = dataset_dir / 'metadata.csv'
times_path = dataset_dir / 'times_ms.npy'

windows = {
    'full_pre_response': (-600.0, -50.0),
    'early_pre_response': (-600.0, -300.0),
    'mid_pre_response': (-300.0, -120.0),
    'late_pre_response': (-120.0, -50.0),
}
alphas = np.logspace(-3, 5, 25)
baseline_columns = ('subject_id', 'difficulty', 'correctness')
targets = ('log_RT_ms', 'RT_ms')
n_splits = 5
seed = 42

print('project_dir =', project_dir)
print('latent_path =', latent_path)
print('dataset_dir =', dataset_dir)


project_dir = /Users/hyijie/Desktop/biiigProject/stage2_YuYNet
latent_path = /Users/hyijie/Desktop/biiigProject/stage2_YuYNet/script_regression/temp_data/latents_full.npz
dataset_dir = /Users/hyijie/Desktop/biiigProject/stage2_YuYNet/script_regression


In [7]:
loaded = np.load(latent_path, allow_pickle=True)

In [11]:
loaded['Z'].shape

(7297, 308, 32)

In [12]:
loaded['times_ms'].shape

(308,)

In [13]:
latent_metadata = pd.DataFrame(loaded['metadata'].item())

In [16]:
latent_metadata

0       sub-STSWD1117
1       sub-STSWD1117
2       sub-STSWD1117
3       sub-STSWD1117
4       sub-STSWD1117
            ...      
7292    sub-STSWD1281
7293    sub-STSWD1281
7294    sub-STSWD1281
7295    sub-STSWD1281
7296    sub-STSWD1281
Name: subj_id, Length: 7297, dtype: str

In [19]:
subj_dummies = pd.get_dummies(latent_metadata["subj_id"], drop_first=True).values.astype(np.float32)
subj_dummies


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(7297, 40), dtype=float32)

In [ ]:
loaded = np.load(latent_path, allow_pickle=True)
latents = loaded['Z']
times_ms = loaded['times_ms'].astype(float)
latent_metadata = pd.DataFrame(loaded['metadata'].item())

source_metadata = pd.read_csv(metadata_path)
source_times = np.load(times_path)

if latents.ndim != 3:
    raise ValueError(f'Expected latent array with shape trial x time x dimension, got {latents.shape}.')
if latents.shape[0] != len(latent_metadata):
    raise ValueError('Latent trial count does not match metadata row count.')
if latents.shape[1] != len(times_ms):
    raise ValueError('Latent time dimension does not match times_ms length.')
if 'RT_ms' not in latent_metadata.columns:
    raise ValueError('Latent metadata must contain RT_ms for RT regression.')
if not np.isfinite(latent_metadata['RT_ms'].to_numpy(dtype=float)).all():
    raise ValueError('RT_ms contains missing or non-finite values.')
if (latent_metadata['RT_ms'].to_numpy(dtype=float) <= 0).any():
    raise ValueError('RT_ms must be positive to compute log_RT_ms.')
if 'trial_id' in latent_metadata.columns and 'trial_id' in source_metadata.columns:
    if latent_metadata['trial_id'].tolist() != source_metadata['trial_id'].tolist():
        raise ValueError('Latent metadata trial_id order does not match dataset metadata.')
if not np.array_equal(times_ms.astype(source_times.dtype, copy=False), source_times):
    raise ValueError('Latent times_ms does not match dataset times_ms.')
if not np.isfinite(latents).all():
    raise ValueError('Latents contain missing or non-finite values.')

quality_report = pd.Series({
    'n_trials': int(latents.shape[0]),
    'n_timepoints': int(latents.shape[1]),
    'hidden_dim': int(latents.shape[2]),
    'rt_missing_count': int(latent_metadata['RT_ms'].isna().sum()),
    'latent_nonfinite_count': int((~np.isfinite(latents)).sum()),
    'dataset_alignment_checked': True,
})

display(quality_report.to_frame('value'))
display(latent_metadata.head())


In [ ]:
metadata = latent_metadata.copy()
metadata['RT_ms'] = pd.to_numeric(metadata['RT_ms'], errors='raise')
metadata['log_RT_ms'] = np.log(metadata['RT_ms'].to_numpy(dtype=float))

available_baseline_columns = [column for column in baseline_columns if column in metadata.columns]
if not available_baseline_columns:
    raise ValueError('None of the requested baseline columns are available in metadata.')

baseline = metadata.loc[:, available_baseline_columns].copy()
categorical_columns = [column for column in baseline.columns if baseline[column].dtype == object or column == 'subject_id']
numeric_columns = [column for column in baseline.columns if column not in categorical_columns]

baseline_parts = []
if numeric_columns:
    baseline_parts.append(baseline[numeric_columns].apply(pd.to_numeric, errors='coerce'))
if categorical_columns:
    baseline_parts.append(pd.get_dummies(baseline[categorical_columns].astype(str), drop_first=False, dtype=float))

baseline_design = pd.concat(baseline_parts, axis=1) if baseline_parts else pd.DataFrame(index=metadata.index)
if baseline_design.isna().any().any():
    baseline_design = baseline_design.fillna(baseline_design.mean(numeric_only=True)).fillna(0.0)
baseline_design = baseline_design.astype(float)

display(pd.DataFrame({'baseline_feature': baseline_design.columns}).head(20))
print('baseline shape =', baseline_design.shape)


In [ ]:
rng = np.random.default_rng(seed)
performance_rows = []
prediction_tables = []
beta_tables = []
feature_quality_rows = []

for window_name, (start_ms, end_ms) in windows.items():
    mask = (times_ms >= start_ms) & (times_ms <= end_ms)
    if int(mask.sum()) < 2:
        raise ValueError(f'Window {start_ms} to {end_ms} ms contains fewer than 2 time points.')

    features = latents[:, mask, :].mean(axis=1)
    hidden = pd.DataFrame(features, columns=[f'hidden_{idx:02d}' for idx in range(features.shape[1])])
    hidden_columns = list(hidden.columns)

    shuffled_indices = rng.permutation(features.shape[0])
    shuffled_hidden = pd.DataFrame(features[shuffled_indices], columns=hidden_columns).add_prefix('shuffled_')

    variances = hidden.var(axis=0).to_numpy(dtype=float)
    feature_quality_rows.append({
        'window': window_name,
        'start_ms': start_ms,
        'end_ms': end_ms,
        'n_timepoints': int(mask.sum()),
        'hidden_dim': int(features.shape[1]),
        'near_zero_variance_dimensions': int(np.sum(variances < 1e-10)),
        'min_feature_variance': float(np.min(variances)),
        'median_feature_variance': float(np.median(variances)),
        'max_feature_variance': float(np.max(variances)),
    })

    designs = {
        'baseline': baseline_design,
        'hidden': hidden,
        'baseline_plus_hidden': pd.concat([baseline_design.reset_index(drop=True), hidden], axis=1),
        'baseline_plus_shuffled_hidden': pd.concat([baseline_design.reset_index(drop=True), shuffled_hidden], axis=1),
    }
    model_hidden_columns = {
        'baseline': [],
        'hidden': hidden_columns,
        'baseline_plus_hidden': hidden_columns,
        'baseline_plus_shuffled_hidden': list(shuffled_hidden.columns),
    }

    for target_name in targets:
        y = metadata[target_name].to_numpy(dtype=float)
        for model_name, design in designs.items():
            folds = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
            predictions = []
            beta_rows = []

            for fold_idx, (train_idx, test_idx) in enumerate(folds.split(design), start=1):
                fold_rng = np.random.default_rng(seed + fold_idx)
                shuffled_train = fold_rng.permutation(train_idx)
                n_alpha_val = max(1, int(round(len(shuffled_train) * 0.2)))
                alpha_val_idx = shuffled_train[:n_alpha_val]
                alpha_train_idx = shuffled_train[n_alpha_val:]
                if len(alpha_train_idx) == 0:
                    alpha_train_idx = train_idx
                    alpha_val_idx = train_idx

                alpha_scores = []
                for alpha in alphas:
                    candidate = Pipeline(steps=[
                        ('scaler', StandardScaler()),
                        ('ridge', Ridge(alpha=float(alpha))),
                    ])
                    candidate.fit(design.iloc[alpha_train_idx], y[alpha_train_idx])
                    alpha_pred = candidate.predict(design.iloc[alpha_val_idx])
                    alpha_scores.append(mean_squared_error(y[alpha_val_idx], alpha_pred))

                alpha = float(alphas[int(np.argmin(alpha_scores))])
                model = Pipeline(steps=[('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
                model.fit(design.iloc[train_idx], y[train_idx])
                pred = model.predict(design.iloc[test_idx])

                for row_idx, pred_value in zip(test_idx, pred):
                    predictions.append({
                        'window': window_name,
                        'target': target_name,
                        'model': model_name,
                        'fold': fold_idx,
                        'row_index': int(row_idx),
                        'y_true': float(y[row_idx]),
                        'y_pred': float(pred_value),
                        'alpha': alpha,
                    })

                coefficients = model.named_steps['ridge'].coef_
                feature_names = list(design.columns)
                hidden_set = set(model_hidden_columns[model_name])
                for feature_name, coefficient in zip(feature_names, coefficients):
                    if feature_name in hidden_set:
                        beta_rows.append({
                            'window': window_name,
                            'target': target_name,
                            'model': model_name,
                            'fold': fold_idx,
                            'feature': feature_name,
                            'coefficient': float(coefficient),
                            'alpha': alpha,
                        })

            predictions_df = pd.DataFrame(predictions)
            beta_df = pd.DataFrame(beta_rows)
            fold_metrics = []

            for fold_idx, fold_df in predictions_df.groupby('fold'):
                fold_y = fold_df['y_true'].to_numpy(dtype=float)
                fold_pred = fold_df['y_pred'].to_numpy(dtype=float)
                fold_metrics.append({
                    'fold': int(fold_idx),
                    'r2': float(r2_score(fold_y, fold_pred)),
                    'rmse': float(np.sqrt(mean_squared_error(fold_y, fold_pred))),
                    'mae': float(mean_absolute_error(fold_y, fold_pred)),
                })

            fold_metrics_df = pd.DataFrame(fold_metrics)
            performance_rows.append({
                'window': window_name,
                'target': target_name,
                'model': model_name,
                'n_trials': int(len(y)),
                'n_features': int(design.shape[1]),
                'mean_cv_r2': float(fold_metrics_df['r2'].mean()),
                'std_cv_r2': float(fold_metrics_df['r2'].std(ddof=0)),
                'pooled_cv_r2': float(r2_score(predictions_df['y_true'], predictions_df['y_pred'])),
                'mean_cv_rmse': float(fold_metrics_df['rmse'].mean()),
                'mean_cv_mae': float(fold_metrics_df['mae'].mean()),
                'mean_alpha': float(predictions_df.groupby('fold')['alpha'].first().mean()),
            })

            prediction_tables.append(predictions_df)
            beta_tables.append(beta_df)

performance = pd.DataFrame(performance_rows)
predictions = pd.concat(prediction_tables, ignore_index=True)
beta = pd.concat(beta_tables, ignore_index=True) if beta_tables else pd.DataFrame()
feature_quality = pd.DataFrame(feature_quality_rows)

display(feature_quality)
display(performance.sort_values(['target', 'window', 'model']).reset_index(drop=True))


In [ ]:
if beta.empty:
    beta_stability = pd.DataFrame(columns=[
        'window', 'target', 'model', 'feature', 'mean_coefficient', 'std_coefficient', 'abs_mean_coefficient', 'sign_consistency'
    ])
else:
    beta_stability = (
        beta.groupby(['window', 'target', 'model', 'feature'])['coefficient']
        .agg(['mean', 'std'])
        .reset_index()
        .rename(columns={'mean': 'mean_coefficient', 'std': 'std_coefficient'})
    )
    beta_stability['std_coefficient'] = beta_stability['std_coefficient'].fillna(0.0)
    beta_stability['abs_mean_coefficient'] = beta_stability['mean_coefficient'].abs()

    sign_consistency = (
        beta.assign(sign=np.sign(beta['coefficient']))
        .groupby(['window', 'target', 'model', 'feature'])['sign']
        .apply(lambda s: max((s > 0).mean(), (s < 0).mean()))
        .reset_index(name='sign_consistency')
    )
    beta_stability = beta_stability.merge(sign_consistency, on=['window', 'target', 'model', 'feature'], how='left')

deltas = []
for (window_name, target_name), group in performance.groupby(['window', 'target']):
    lookup = group.set_index('model')
    baseline_r2 = float(lookup.loc['baseline', 'mean_cv_r2']) if 'baseline' in lookup.index else np.nan
    hidden_r2 = float(lookup.loc['hidden', 'mean_cv_r2']) if 'hidden' in lookup.index else np.nan
    baseline_plus_hidden_r2 = float(lookup.loc['baseline_plus_hidden', 'mean_cv_r2']) if 'baseline_plus_hidden' in lookup.index else np.nan
    baseline_plus_shuffled_hidden_r2 = float(lookup.loc['baseline_plus_shuffled_hidden', 'mean_cv_r2']) if 'baseline_plus_shuffled_hidden' in lookup.index else np.nan
    deltas.append({
        'window': window_name,
        'target': target_name,
        'baseline_r2': baseline_r2,
        'hidden_r2': hidden_r2,
        'baseline_plus_hidden_r2': baseline_plus_hidden_r2,
        'baseline_plus_shuffled_hidden_r2': baseline_plus_shuffled_hidden_r2,
        'hidden_minus_baseline_r2': hidden_r2 - baseline_r2,
        'incremental_minus_baseline_r2': baseline_plus_hidden_r2 - baseline_r2,
        'incremental_minus_shuffled_r2': baseline_plus_hidden_r2 - baseline_plus_shuffled_hidden_r2,
    })

deltas = pd.DataFrame(deltas)
display(deltas.sort_values(['target', 'window']).reset_index(drop=True))
display(
    beta_stability.sort_values(['target', 'window', 'abs_mean_coefficient'], ascending=[True, True, False])
    .groupby(['target', 'window'])
    .head(10)
    .reset_index(drop=True)
)


In [ ]:
log_deltas = deltas[deltas['target'] == 'log_RT_ms'].copy()
best_log_rt_row = log_deltas.sort_values('incremental_minus_baseline_r2', ascending=False).iloc[0]
best_window = best_log_rt_row['window']

summary_table = pd.DataFrame([
    {
        'best_window': best_window,
        'incremental_minus_baseline_r2': float(best_log_rt_row['incremental_minus_baseline_r2']),
        'incremental_minus_shuffled_r2': float(best_log_rt_row['incremental_minus_shuffled_r2']),
    }
])
display(summary_table)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(log_deltas))
plot_data = log_deltas.set_index('window').loc[list(windows.keys())].reset_index()
ax.axhline(0.0, color='black', linewidth=1.0, alpha=0.4)
ax.bar(x - 0.18, plot_data['incremental_minus_baseline_r2'], width=0.36, label='hidden added to baseline')
ax.bar(x + 0.18, plot_data['incremental_minus_shuffled_r2'], width=0.36, label='real hidden vs shuffled')
ax.set_xticks(x)
ax.set_xticklabels(plot_data['window'], rotation=25, ha='right')
ax.set_ylabel('mean CV R2 difference')
ax.set_title('Incremental value of hidden states for log RT')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
best_predictions = predictions[
    (predictions['window'] == best_window)
    & (predictions['target'] == 'log_RT_ms')
    & (predictions['model'] == 'baseline_plus_hidden')
].copy()

best_predictions = best_predictions.sort_values('row_index').reset_index(drop=True)
display(best_predictions.head())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].scatter(best_predictions['y_true'], best_predictions['y_pred'], s=12, alpha=0.45)
line_min = min(best_predictions['y_true'].min(), best_predictions['y_pred'].min())
line_max = max(best_predictions['y_true'].max(), best_predictions['y_pred'].max())
axes[0].plot([line_min, line_max], [line_min, line_max], color='black', linewidth=1.0)
axes[0].set_xlabel('Observed log RT')
axes[0].set_ylabel('Predicted log RT')
axes[0].set_title(f'Best model predictions: {best_window}')

top_beta = (
    beta_stability[
        (beta_stability['window'] == best_window)
        & (beta_stability['target'] == 'log_RT_ms')
        & (beta_stability['model'] == 'baseline_plus_hidden')
    ]
    .sort_values('abs_mean_coefficient', ascending=False)
    .head(12)
    .sort_values('mean_coefficient')
)

axes[1].barh(top_beta['feature'], top_beta['mean_coefficient'])
axes[1].set_xlabel('Mean ridge coefficient across folds')
axes[1].set_title('Top hidden dimensions in best window')

plt.tight_layout()
plt.show()

display(top_beta.reset_index(drop=True))
